# Praktikum Pengolahan Citra Digital: Sub-CPMK 4
## Perbaikan Tampilan Citra dengan Transformasi Intensitas dan Histogram

Setelah praktikum ini, mahasiswa dapat mengenali masalah kecerahan atau kontras, mencoba gamma, histogram equalization, dan CLAHE, lalu memilih hasil berdasarkan bagian gambar yang ingin diperjelas.

**Pola kerja:** prediksi, jalankan, amati, lalu jelaskan.

## Aturan latihan

Latihan wajib mengulang langkah yang sudah didemonstrasikan. Mahasiswa hanya mengubah parameter yang ditandai. Penggunaan foto lain tersedia setelah latihan wajib selesai.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11
print("OpenCV:", cv2.__version__)

In [ ]:
ASSET_DIR = Path("assets/subcpmk4")

def gamma_correction(image, gamma):
    normalized = image.astype(np.float32) / 255.0
    result = 255 * np.power(normalized, gamma)
    return np.clip(result, 0, 255).astype(np.uint8)

def pencahayaan_tidak_merata(image, kiri=0.30, kanan=1.08):
    h, w = image.shape
    gradient = np.tile(np.linspace(kiri, kanan, w, dtype=np.float32), (h, 1))
    return np.clip(image.astype(np.float32) * gradient, 0, 255).astype(np.uint8)

demo_path = ASSET_DIR / "demo_cahaya_tidak_merata.png"
latihan_path = ASSET_DIR / "latihan_cahaya_tidak_merata.png"

if demo_path.exists() and latihan_path.exists():
    demo = cv2.imread(str(demo_path), cv2.IMREAD_GRAYSCALE)
    latihan = cv2.imread(str(latihan_path), cv2.IMREAD_GRAYSCALE)
else:
    demo = pencahayaan_tidak_merata(data.camera().astype(np.uint8))
    latihan = pencahayaan_tidak_merata(data.coins().astype(np.uint8), 0.35, 1.05)

print("Demo:", demo.shape, demo.dtype)
print("Latihan:", latihan.shape, latihan.dtype)

In [ ]:
def tampilkan(image, judul="Citra", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(judul)
    ax.axis("off")

def tampilkan_dengan_histogram(image, judul="Citra"):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    tampilkan(image, judul, ax[0])
    ax[1].hist(image.ravel(), 256, [0, 256], color="#153A70")
    ax[1].set_title("Histogram")
    ax[1].set_xlim(0, 255)
    ax[1].set_xlabel("Intensitas")
    ax[1].set_ylabel("Jumlah piksel")
    plt.tight_layout()
    plt.show()

def bandingkan(images, titles, figsize=(15, 4)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    for ax, image, title in zip(axes, images, titles):
        tampilkan(image, title, ax)
    plt.tight_layout()
    plt.show()

## 1. Diagnosis citra sebelum pengolahan

Perhatikan gambar dan histogram. Tentukan wilayah yang ingin diperjelas. Histogram menunjukkan distribusi intensitas, tetapi tidak menunjukkan posisi piksel.

In [ ]:
tampilkan_dengan_histogram(demo, "Citra demo")

**Prediksi sebelum lanjut:** jika semua piksel dibuat lebih terang, apa yang mungkin terjadi pada bagian yang sudah terang?

Jawaban yang diharapkan: nilai dapat mencapai 255 dan kehilangan perbedaan karena clipping.

In [ ]:
contoh = np.array([230, 250], dtype=np.int16)
setelah_tambah = np.clip(contoh + 40, 0, 255)
setelah_digelapkan = np.clip(setelah_tambah - 40, 0, 255)
print("Awal             :", contoh)
print("Tambah 40        :", setelah_tambah)
print("Kemudian kurangi :", setelah_digelapkan)

## 2. Transformasi intensitas singkat

Setiap piksel input `r` dipetakan menjadi output `s`. Negatif membalik terang dan gelap. Transformasi log menonjolkan perbedaan pada intensitas rendah. Keduanya menjadi pengantar sebelum eksperimen gamma.

In [ ]:
negatif = 255 - demo
c = 255 / np.log1p(255)
log_result = np.clip(c * np.log1p(demo.astype(np.float32)), 0, 255).astype(np.uint8)
bandingkan([demo, negatif, log_result], ["Asli", "Negatif", "Log"], figsize=(12, 4))

## 3. Gamma correction

Notebook ini memakai rumus `s = 255 × (r/255)^gamma`. Gamma di bawah 1 mencerahkan nilai tengah, gamma 1 mempertahankan citra, dan gamma di atas 1 menggelapkan.

**Prediksi dahulu:** dari gamma 0,5; 1; dan 2, mana yang paling mungkin memperjelas citra gelap?

In [ ]:
gamma_values = [0.5, 0.75, 1.0, 2.0]
gamma_images = [gamma_correction(demo, g) for g in gamma_values]
bandingkan(gamma_images, [f"Gamma {g}" for g in gamma_values], figsize=(15, 4))

## 4. Histogram equalization global

Equalization global membentuk pemetaan dari histogram kumulatif seluruh citra. Metode ini dapat memperlebar rentang intensitas, tetapi satu pemetaan berlaku untuk seluruh lokasi.

In [ ]:
equalized_demo = cv2.equalizeHist(demo)
tampilkan_dengan_histogram(equalized_demo, "Equalization global")

## 5. CLAHE

CLAHE bekerja pada pembagian wilayah citra dan membatasi penguatan kontras. `tileGridSize=(8, 8)` berarti citra dibagi menjadi delapan bagian pada setiap arah. Pada latihan utama, grid tetap dan hanya `clipLimit` yang diubah.

In [ ]:
clahe_results = []
clip_values = [1.0, 2.0, 4.0]
for clip in clip_values:
    operator = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    clahe_results.append(operator.apply(demo))
bandingkan(clahe_results, [f"clipLimit {v}" for v in clip_values], figsize=(13, 4))

## 6. Perbandingan metode

Semua hasil berikut berasal dari citra input yang sama. Periksa area gelap, detail di area terang, dan derau yang ikut terlihat.

In [ ]:
# Tiga metode yang akan dibandingkan. Semua dihitung dari citra demo yang sama.
hasil_gamma = gamma_correction(demo, gamma=0.65)
hasil_equalization = cv2.equalizeHist(demo)

objek_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
hasil_clahe = objek_clahe.apply(demo)

bandingkan(
    [demo, hasil_gamma, hasil_equalization, hasil_clahe],
    ["Asli", "Gamma 0,65", "Equalization global", "CLAHE"],
    figsize=(16, 4)
)

## 7. Latihan wajib: ulangi demonstrasi

Gunakan citra `latihan`. Urutan dan kode sama dengan demonstrasi. Mahasiswa hanya memilih gamma dan `clipLimit` dari nilai yang tersedia.

In [ ]:
# LATIHAN WAJIB
# Ikuti langkah yang sama seperti demonstrasi. Ubah hanya nilai yang ditandai.

# 1. Coba tiga gamma yang sudah disediakan.
gamma_dicoba = [0.50, 0.70, 1.00]
hasil_gamma_latihan = [gamma_correction(latihan, g) for g in gamma_dicoba]
bandingkan(
    [latihan] + hasil_gamma_latihan,
    ["Asli"] + [f"Gamma {g}" for g in gamma_dicoba],
    figsize=(15, 4)
)

# 2. Pilih satu gamma dari daftar di atas.
# TODO mahasiswa: ganti angka berikut dengan pilihanmu: 0.50, 0.70, atau 1.00.
gamma_pilihan = 0.70
hasil_gamma_pilihan = gamma_correction(latihan, gamma_pilihan)

# 3. Ulangi equalization global dan CLAHE seperti pada demonstrasi.
hasil_equalization_latihan = cv2.equalizeHist(latihan)

# TODO mahasiswa: coba clipLimit 1.0 dan 2.0, lalu pilih salah satunya.
clip_pilihan = 2.0
clahe_latihan = cv2.createCLAHE(clipLimit=clip_pilihan, tileGridSize=(8, 8))
hasil_clahe_latihan = clahe_latihan.apply(latihan)

# 4. Bandingkan seluruh hasil dari input yang sama.
bandingkan(
    [latihan, hasil_gamma_pilihan, hasil_equalization_latihan, hasil_clahe_latihan],
    ["Asli", f"Gamma {gamma_pilihan}", "Equalization", f"CLAHE {clip_pilihan}"],
    figsize=(16, 4)
)

### Catatan hasil

Isi setelah menjalankan latihan.

| Metode | Parameter | Detail yang lebih terlihat | Efek yang kurang diinginkan |
|---|---|---|---|
| Gamma | ... | ... | ... |
| Equalization global | tidak ada | ... | ... |
| CLAHE | ... | ... | ... |

**Pilihan hasil:** ...

**Alasan:** sebutkan bagian gambar yang menjadi bukti. Cukup dua sampai empat kalimat.

## 8. Tambahan opsional

Bagian ini dikerjakan setelah latihan wajib selesai. Mahasiswa dapat memakai gambar lain atau foto sendiri, lalu mengulang blok perbandingan yang sama.

In [ ]:
# OPSIONAL SETELAH LATIHAN WAJIB SELESAI
# Unggah gambar sendiri di Google Colab, ubah path, lalu ulangi blok perbandingan.
# Gambar sebaiknya grayscale atau akan dikonversi ke grayscale.

# from google.colab import files
# uploaded = files.upload()
# nama_file = next(iter(uploaded))
# foto = cv2.imread(nama_file, cv2.IMREAD_GRAYSCALE)
# tampilkan_dengan_histogram(foto, "Foto tambahan")

## Pengumpulan

Kumpulkan notebook yang sudah dijalankan. Pastikan perbandingan empat gambar, tabel pengamatan, pilihan hasil, dan alasannya sudah terlihat.